# Exploratory Data Analysis — Retail Transaction Dataset

**Prepared by:** Shailly Maurya  
**Dataset:** Retail Large Dataset  
**Records:** 100,000  
**Variables:** 18 original variables

## Project objective
This notebook performs a professional exploratory data analysis of a large-scale retail transaction dataset. It covers data quality, descriptive statistics, univariate/bivariate/multivariate analysis, outlier detection, correlation, skewness, kurtosis, and business-oriented insights.

> **Analytical principle:** descriptive associations are not treated as causal relationships. Outliers are investigated rather than automatically deleted.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
sns.set_theme(style="whitegrid")

df = pd.read_csv("retail_large_dataset.csv")
df.head()


## 1. Dataset structure and quality

In [ ]:
print("Shape:", df.shape)
display(df.head())
display(df.dtypes.to_frame("Data Type"))
display(df.describe(include="all").T)


In [ ]:
quality = pd.DataFrame({
    "missing_values": df.isna().sum(),
    "missing_%": df.isna().mean()*100,
    "unique_values": df.nunique()
})
display(quality)
print("Duplicate rows:", df.duplicated().sum())


## 2. Data preparation and derived metrics

In [ ]:
df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")
df["gross_amount"] = df["product_price"] * df["quantity"]
df["discount_amount"] = df["gross_amount"] - df["final_price"]
df["order_month"] = df["order_date"].dt.to_period("M").astype(str)
df["order_year"] = df["order_date"].dt.year
df["return_flag"] = df["return_status"].eq("Yes").astype(int)

# Validate the final-price formula used in the dataset
expected_final = df["product_price"] * df["quantity"] * (1 - df["discount_percentage"]/100)
print("Final-price formula mismatches:", (abs(df["final_price"]-expected_final) > 0.01).sum())


## 3. Descriptive statistics, skewness and kurtosis

In [ ]:
numeric_cols = ["age","product_price","quantity","discount_percentage","final_price","delivery_days"]
summary = df[numeric_cols].describe().T
summary["median"] = df[numeric_cols].median()
summary["skewness"] = df[numeric_cols].skew()
summary["kurtosis"] = df[numeric_cols].kurtosis()
display(summary[["count","mean","median","std","min","25%","50%","75%","max","skewness","kurtosis"]])


## 4. Univariate categorical analysis

In [ ]:
categorical_cols = ["gender","city","state","customer_segment","product_category",
                    "product_subcategory","payment_method","shipping_type","return_status"]

for col in categorical_cols:
    print(f"\n--- {col} ---")
    display(df[col].value_counts(dropna=False).to_frame("Count").assign(Percentage=lambda x: x["Count"]/len(df)*100))


## 5. Distribution visualizations

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, col in zip(axes.ravel(), numeric_cols):
    ax.hist(df[col], bins=40)
    ax.set_title(f"Distribution of {col}")
    ax.set_xlabel(col)
    ax.set_ylabel("Frequency")
plt.tight_layout()
plt.show()


## 6. Outlier detection — IQR and Z-score

In [ ]:
outlier_results = []
for col in numeric_cols:
    x = df[col]
    q1, q3 = x.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr
    iqr_mask = (x < lower) | (x > upper)
    z_mask = np.abs(stats.zscore(x, nan_policy="omit")) > 3
    outlier_results.append({
        "Variable": col,
        "IQR Outliers": int(iqr_mask.sum()),
        "IQR %": iqr_mask.mean()*100,
        "Z > 3 Outliers": int(z_mask.sum()),
        "Z > 3 %": z_mask.mean()*100,
        "IQR Lower": lower,
        "IQR Upper": upper
    })
outlier_table = pd.DataFrame(outlier_results)
display(outlier_table)


In [ ]:
plt.figure(figsize=(10,5))
plt.boxplot(df["final_price"], vert=False)
plt.title("Final Price — Outlier Review")
plt.xlabel("Final Price")
plt.show()


## 7. Bivariate analysis

In [ ]:
category_summary = df.groupby("product_category").agg(
    Transactions=("order_id","count"),
    Revenue=("final_price","sum"),
    Avg_Order_Value=("final_price","mean"),
    Return_Rate=("return_flag","mean")
)
category_summary["Return_Rate"] *= 100
display(category_summary.sort_values("Revenue", ascending=False))


In [ ]:
plt.figure(figsize=(9,5))
sns.boxplot(data=df, x="customer_segment", y="final_price")
plt.title("Final Transaction Value by Customer Segment")
plt.show()


In [ ]:
plt.figure(figsize=(8,6))
sns.scatterplot(data=df, x="product_price", y="final_price", alpha=.15, s=15)
plt.title("Product Price vs Final Price")
plt.show()


## 8. Pearson correlation

In [ ]:
corr = df[numeric_cols].corr(method="pearson")
display(corr)
plt.figure(figsize=(9,7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Pearson Correlation Matrix")
plt.show()


## 9. Multivariate analysis

In [ ]:
plt.figure(figsize=(11,6))
sns.scatterplot(
    data=df.sample(min(15000, len(df)), random_state=42),
    x="discount_percentage",
    y="final_price",
    hue="product_category",
    alpha=.35
)
plt.title("Discount Percentage vs Final Price by Product Category")
plt.show()


In [ ]:
segment_category = pd.pivot_table(
    df, values="final_price", index="customer_segment",
    columns="product_category", aggfunc="mean"
)
display(segment_category.round(2))


## 10. Delivery and returns

In [ ]:
shipping_summary = df.groupby("shipping_type").agg(
    Transactions=("order_id","count"),
    Avg_Delivery_Days=("delivery_days","mean"),
    Return_Rate=("return_flag","mean"),
    Revenue=("final_price","sum")
)
shipping_summary["Return_Rate"] *= 100
display(shipping_summary)


In [ ]:
return_by_delivery = df.groupby("delivery_days")["return_flag"].mean()*100
display(return_by_delivery.to_frame("Return Rate %"))


## 11. Time analysis

In [ ]:
monthly = df.groupby("order_month").agg(
    Transactions=("order_id","count"),
    Revenue=("final_price","sum"),
    Avg_Order_Value=("final_price","mean"),
    Return_Rate=("return_flag","mean")
).reset_index()
monthly["Return_Rate"] *= 100
display(monthly.head())

plt.figure(figsize=(13,5))
plt.plot(monthly["order_month"], monthly["Revenue"])
plt.xticks(rotation=60)
plt.title("Monthly Revenue Trend")
plt.xlabel("Order Month")
plt.ylabel("Revenue")
plt.show()


## 12. Key business findings

Based on the dataset analyzed:

1. **Scale and completeness:** The dataset contains 100,000 transaction records and 18 original variables, with no missing values and no exact duplicate rows.
2. **Returns:** The overall observed return rate is approximately 14.8%.
3. **Category performance:** Home & Kitchen generates the highest total revenue among the six product categories in this dataset.
4. **Customer segments:** Average transaction value differs only modestly between Premium, Regular and New segments; segment labels should therefore not be treated as proof of materially different customer value without further testing.
5. **Pricing relationship:** Product price has a strong positive Pearson correlation with final transaction value, while discount percentage has a modest negative association.
6. **Outliers:** Final transaction value contains identifiable high-value observations. These should be investigated as potential premium purchases rather than automatically deleted.
7. **Delivery:** Express and Standard shipping have very similar average delivery times and return rates in this dataset, so the descriptive evidence does not indicate a large operational gap between the two shipping types.
8. **Causality limitation:** These are observational relationships. The analysis does not establish that discounts, delivery times or customer segments cause changes in returns or spending.


## 13. Conclusion

The analysis provides a structured view of retail transaction behavior across customers, products, pricing, discounts, payment methods, shipping and returns. The strongest quantitative relationship is between product price and final transaction value. The dataset is complete and internally consistent on the core price formula, while final transaction value contains a relatively small set of high-value observations that warrant business review.

### Recommended next steps
- Investigate high-value transactions and their product/subcategory mix.
- Segment return analysis by category, subcategory, city and delivery time.
- Examine discount effectiveness against revenue and return behavior.
- Add statistical significance testing or predictive modeling if the project is extended beyond EDA.
